In [ ]:
# !pip install pandas openpyxl earthengine-api geopandas

import ee
import pandas as pd
import os
import json
import pickle
from datetime import datetime, UTC
import time
import concurrent.futures
from threading import Lock
import logging
from logging.handlers import RotatingFileHandler
import traceback
from pathlib import Path
import sys

# ==============================================================================
# 1. CONFIGURACIÓN INICIAL MEJORADA
# ==============================================================================

# --- BASE PATHS (MODIFICA ESTAS RUTAS SEGÚN TU SISTEMA) ---
BASE_DIR = Path(r"E:\OneDrive - Colorado School of Mines\projects_2025\chile_groundwater")
input_excel_path = BASE_DIR / "data" / "database_2025" / "groundwater_chile_and_elevation_dataset_2025.xlsx"
output_folder = BASE_DIR / "python_code" / "results"
logs_folder = output_folder / "logs"
checkpoints_folder = output_folder / "checkpoints"

# Crear todas las carpetas necesarias
for folder in [output_folder, logs_folder, checkpoints_folder]:
    folder.mkdir(parents=True, exist_ok=True)

# --- ARCHIVOS DE SALIDA ---
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_excel_filename = f"resultados_gee_chile_{timestamp}.xlsx"
output_excel_path = output_folder / output_excel_filename
checkpoint_file = checkpoints_folder / f"checkpoint_{timestamp}.pkl"
progress_file = output_folder / f"progress_{timestamp}.json"

# --- CONFIGURACIÓN DE LOGGING AVANZADA ---
def setup_advanced_logging():
    """Configura sistema de logging robusto con múltiples handlers"""
    
    # Crear logger principal
    logger = logging.getLogger('gee_extraction')
    logger.setLevel(logging.DEBUG)
    
    # Evitar duplicados
    if logger.handlers:
        logger.handlers.clear()
    
    # Formato detallado
    detailed_formatter = logging.Formatter(
        '%(asctime)s | %(levelname)-8s | %(funcName)-20s | %(lineno)-4d | %(message)s',
        datefmt='%Y-%m-%d %H:%M:%S'
    )
    
    # Formato simple para consola
    console_formatter = logging.Formatter(
        '%(asctime)s | %(levelname)-8s | %(message)s',
        datefmt='%H:%M:%S'
    )
    
    # 1. Handler para archivo principal (rotativo)
    main_log_file = logs_folder / f"gee_extraction_{timestamp}.log"
    file_handler = RotatingFileHandler(
        main_log_file, 
        maxBytes=50*1024*1024,  # 50MB
        backupCount=5,
        encoding='utf-8'
    )
    file_handler.setLevel(logging.DEBUG)
    file_handler.setFormatter(detailed_formatter)
    logger.addHandler(file_handler)
    
    # 2. Handler para errores críticos
    error_log_file = logs_folder / f"errors_{timestamp}.log"
    error_handler = RotatingFileHandler(
        error_log_file,
        maxBytes=10*1024*1024,  # 10MB
        backupCount=3,
        encoding='utf-8'
    )
    error_handler.setLevel(logging.ERROR)
    error_handler.setFormatter(detailed_formatter)
    logger.addHandler(error_handler)
    
    # 3. Handler para consola
    console_handler = logging.StreamHandler(sys.stdout)
    console_handler.setLevel(logging.INFO)
    console_handler.setFormatter(console_formatter)
    logger.addHandler(console_handler)
    
    # 4. Handler para progreso (archivo separado)
    progress_log_file = logs_folder / f"progress_{timestamp}.log"
    progress_handler = RotatingFileHandler(
        progress_log_file,
        maxBytes=5*1024*1024,  # 5MB
        backupCount=2,
        encoding='utf-8'
    )
    progress_handler.setLevel(logging.INFO)
    progress_handler.setFormatter(console_formatter)
    
    # Solo agregar mensajes de progreso a este handler
    class ProgressFilter(logging.Filter):
        def filter(self, record):
            return 'PROGRESS' in record.getMessage() or 'CHECKPOINT' in record.getMessage()
    
    progress_handler.addFilter(ProgressFilter())
    logger.addHandler(progress_handler)
    
    return logger

# Configurar logging
logger = setup_advanced_logging()

# --- CONFIGURACIÓN DE PROCESAMIENTO OPTIMIZADA ---
import multiprocessing
CPU_COUNT = multiprocessing.cpu_count()

# Configuración agresiva para máximo rendimiento
BATCH_SIZE = min(50, CPU_COUNT * 4)  # Lotes más grandes
SAVE_INTERVAL = 5  # Guardar muy frecuentemente
MAX_WORKERS = min(CPU_COUNT, 8)  # Usar todos los CPUs disponibles pero no más de 8
TIMEOUT_SECONDS = 60  # Timeout más largo
MAX_RETRIES = 5  # Más reintentos
PARALLEL_CHUNKS = MAX_WORKERS * 2  # Chunks para procesamiento paralelo

logger.info(f"CONFIGURACIÓN DE PROCESAMIENTO:")
logger.info(f"  - CPUs detectados: {CPU_COUNT}")
logger.info(f"  - Workers paralelos: {MAX_WORKERS}")
logger.info(f"  - Tamaño de lote: {BATCH_SIZE}")
logger.info(f"  - Intervalo de guardado: {SAVE_INTERVAL}")
logger.info(f"  - Chunks paralelos: {PARALLEL_CHUNKS}")

# Locks para operaciones thread-safe
save_lock = Lock()
progress_lock = Lock()

# --- AUTENTICACIÓN CON GOOGLE EARTH ENGINE ---
# Set EE_SERVICE_ACCOUNT and EE_KEY_FILE to your own Earth Engine service account and its JSON key
service_account_email = os.environ["EE_SERVICE_ACCOUNT"]
json_key_path = Path(os.environ["EE_KEY_FILE"])

logger.info("Iniciando autenticación con Google Earth Engine...")

if not json_key_path.exists():
    raise FileNotFoundError(f"El archivo de clave JSON no se encontró: {json_key_path}")

try:
    credentials = ee.ServiceAccountCredentials(service_account_email, str(json_key_path))
    ee.Initialize(credentials)
    logger.info("✓ Autenticación con cuenta de servicio de GEE exitosa.")
except Exception as auth_error:
    logger.error(f"✗ Error al autenticarse con cuenta de servicio: {auth_error}")
    raise

# ==============================================================================
# 2. FUNCIONES DE CHECKPOINT Y PROGRESO
# ==============================================================================

class ProgressTracker:
    """Clase para rastrear y guardar progreso detallado"""
    
    def __init__(self, progress_file_path):
        self.progress_file = Path(progress_file_path)
        self.data = {
            'session_start': datetime.now().isoformat(),
            'last_update': None,
            'total_points': 0,
            'processed_points': 0,
            'successful_points': 0,
            'failed_points': 0,
            'current_batch': 0,
            'processing_rate': 0.0,
            'estimated_completion': None,
            'errors': [],
            'dataset_stats': {},
            'last_processed_id': None
        }
        self.start_time = time.time()
        
    def update(self, **kwargs):
        """Actualiza el progreso con nuevos datos"""
        with progress_lock:
            for key, value in kwargs.items():
                if key in self.data:
                    self.data[key] = value
            
            self.data['last_update'] = datetime.now().isoformat()
            
            # Calcular tasa de procesamiento
            elapsed_time = time.time() - self.start_time
            if elapsed_time > 0 and self.data['processed_points'] > 0:
                self.data['processing_rate'] = self.data['processed_points'] / elapsed_time
                
                # Estimar tiempo de finalización
                remaining_points = self.data['total_points'] - self.data['processed_points']
                if self.data['processing_rate'] > 0:
                    remaining_seconds = remaining_points / self.data['processing_rate']
                    completion_time = datetime.now() + pd.Timedelta(seconds=remaining_seconds)
                    self.data['estimated_completion'] = completion_time.isoformat()
            
            self.save()
    
    def add_error(self, error_info):
        """Agrega información de error"""
        with progress_lock:
            self.data['errors'].append({
                'timestamp': datetime.now().isoformat(),
                'error': str(error_info),
                'traceback': traceback.format_exc()
            })
            self.data['failed_points'] += 1
            self.save()
    
    def save(self):
        """Guarda el progreso a archivo"""
        try:
            with open(self.progress_file, 'w', encoding='utf-8') as f:
                json.dump(self.data, f, indent=2, ensure_ascii=False)
        except Exception as e:
            logger.error(f"Error guardando progreso: {e}")
    
    def load(self):
        """Carga progreso existente"""
        if self.progress_file.exists():
            try:
                with open(self.progress_file, 'r', encoding='utf-8') as f:
                    self.data.update(json.load(f))
                return True
            except Exception as e:
                logger.error(f"Error cargando progreso: {e}")
        return False

def save_checkpoint(processed_results, current_index, checkpoint_file_path):
    """Guarda checkpoint completo del procesamiento"""
    checkpoint_data = {
        'timestamp': datetime.now().isoformat(),
        'current_index': current_index,
        'processed_results': processed_results,
        'total_processed': len(processed_results),
        'last_id': processed_results[-1]['ID'] if processed_results else None
    }
    
    try:
        with open(checkpoint_file_path, 'wb') as f:
            pickle.dump(checkpoint_data, f)
        logger.info(f"CHECKPOINT: Guardado en {checkpoint_file_path} (Índice: {current_index})")
        return True
    except Exception as e:
        logger.error(f"Error guardando checkpoint: {e}")
        return False

def load_checkpoint(checkpoint_file_path):
    """Carga checkpoint existente"""
    if Path(checkpoint_file_path).exists():
        try:
            with open(checkpoint_file_path, 'rb') as f:
                checkpoint_data = pickle.load(f)
            logger.info(f"CHECKPOINT: Cargado desde {checkpoint_file_path}")
            logger.info(f"  - Puntos procesados: {checkpoint_data['total_processed']}")
            logger.info(f"  - Último índice: {checkpoint_data['current_index']}")
            logger.info(f"  - Último ID: {checkpoint_data['last_id']}")
            return checkpoint_data
        except Exception as e:
            logger.error(f"Error cargando checkpoint: {e}")
    return None

# ==============================================================================
# 3. CARGA Y PREPARACIÓN DE DATOS
# ==============================================================================

logger.info(f"Cargando datos de puntos desde: {input_excel_path}")
if not input_excel_path.exists():
    raise FileNotFoundError(f"Archivo no encontrado: {input_excel_path}")

try:
    df_points = pd.read_excel(input_excel_path)
    # df_points = df_points.head(20).copy()  # Descomenta para probar con pocos puntos
    logger.info(f"✓ Cargados {len(df_points)} puntos desde Excel")
except Exception as e:
    logger.error(f"✗ Error al leer el archivo Excel: {e}")
    raise ValueError(f"Error al leer el archivo Excel: {e}")

# Verificar columnas esenciales
required_cols = ['ID', 'Date_String', 'Latitude_GCS_WGS_1984', 'Longitude_GCS_WGS_1984']
missing_cols = [col for col in required_cols if col not in df_points.columns]
if missing_cols:
    raise ValueError(f"Columnas requeridas no encontradas: {missing_cols}")

logger.info("✓ Todas las columnas requeridas están presentes")

# Convertir fechas
df_points['Date_String_dt'] = pd.to_datetime(df_points['Date_String'], errors='coerce')
logger.info(f"✓ Fechas procesadas - Válidas: {df_points['Date_String_dt'].notna().sum()}")

# Inicializar tracker de progreso
progress_tracker = ProgressTracker(progress_file)
progress_tracker.update(total_points=len(df_points))

# Verificar checkpoint existente
checkpoint_data = load_checkpoint(checkpoint_file)
start_index = 0
final_results_list = []

if checkpoint_data:
    final_results_list = checkpoint_data['processed_results']
    start_index = checkpoint_data['current_index'] + 1
    logger.info(f"CHECKPOINT: Reanudando desde índice {start_index}")
    progress_tracker.update(
        processed_points=len(final_results_list),
        last_processed_id=checkpoint_data['last_id']
    )

# Determinar puntos a procesar
df_points_to_process = df_points.iloc[start_index:].copy()
total_points_to_process = len(df_points_to_process)

logger.info(f"CONFIGURACIÓN FINAL:")
logger.info(f"  - Total de puntos en dataset: {len(df_points)}")
logger.info(f"  - Puntos ya procesados: {len(final_results_list)}")
logger.info(f"  - Puntos por procesar: {total_points_to_process}")
logger.info(f"  - Índice de inicio: {start_index}")

# ==============================================================================
# 4. METADATOS DE DATASETS DE GEE (MISMO QUE ANTES)
# ==============================================================================

gee_datasets_metadata = [
    # --- GRUPO 1: MODELOS DE ELEVACIÓN DIGITAL (DEMs) ---
    {
        'name': 'Copernicus DEM GLO-30', 
        'id': 'COPERNICUS/DEM/GLO30', 
        'type': 'ImageCollection', 
        'is_temporal': False,
        'availability': '2021-04-22', 
        'pixel_size': 30, 
        'prefix': 'cop_dem_30',
        'bands': [{'band_name': 'DEM', 'unit': 'm', 'description': 'Elevation above EGM2008 geoid (Digital Surface Model)'}],
        'priority': 1,
        'processing_method': 'mosaic'
    },
    
    # --- GRUPO 2: COBERTURA DE SUELO DE RESOLUCIÓN MEDIA (500m - 1km) ---
    {
        'name': 'MODIS Land Cover Type Yearly Global 500m', 
        'id': 'MODIS/061/MCD12Q1', 
        'type': 'ImageCollection', 
        'is_temporal': True,
        'availability': '2001-01-01 a presente', 
        'pixel_size': 500, 
        'prefix': 'modis_lc',
        'bands': [
            {'band_name': 'LC_Type1', 'unit': 'class', 'description': 'Land Cover Type 1 (IGBP)'},
            {'band_name': 'LC_Type2', 'unit': 'class', 'description': 'Land Cover Type 2 (UMD)'},
            {'band_name': 'LC_Type3', 'unit': 'class', 'description': 'Land Cover Type 3 (LAI)'},
            {'band_name': 'LC_Type4', 'unit': 'class', 'description': 'Land Cover Type 4 (BGC)'},
            {'band_name': 'LC_Type5', 'unit': 'class', 'description': 'Land Cover Type 5 (PFT)'}
        ],
        'priority': 2,
        'processing_method': 'closest_temporal'
    },

    # --- GRUPO 3: VARIABLES CLIMÁTICAS Y BIO-CLIMÁTICAS ---
    {
        'name': 'TerraClimate Monthly', 
        'id': 'IDAHO_EPSCOR/TERRACLIMATE', 
        'type': 'ImageCollection', 
        'is_temporal': True,
        'availability': '1958-01-01 a presente', 
        'pixel_size': 4000, 
        'prefix': 'terraclim',
        'bands': [
            {'band_name': 'pr', 'unit': 'mm', 'description': 'Precipitation accumulation'},
            {'band_name': 'tmmn', 'unit': '°C', 'description': 'Minimum temperature (Scale 0.1)'},
            {'band_name': 'tmmx', 'unit': '°C', 'description': 'Maximum temperature (Scale 0.1)'},
            {'band_name': 'aet', 'unit': 'mm', 'description': 'Actual evapotranspiration (Scale 0.1)'},
            {'band_name': 'pet', 'unit': 'mm', 'description': 'Reference evapotranspiration (Scale 0.1)'},
            {'band_name': 'def', 'unit': 'mm', 'description': 'Climate water deficit (Scale 0.1)'},
            {'band_name': 'pdsi', 'unit': 'unitless', 'description': 'Palmer Drought Severity Index'},
            {'band_name': 'srad', 'unit': 'W/m^2', 'description': 'Downward surface shortwave radiation'},
            {'band_name': 'vpd', 'unit': 'kPa', 'description': 'Vapor pressure deficit (Scale 0.01)'},
            {'band_name': 'vs', 'unit': 'm/s', 'description': 'Wind speed at 10m (Scale 0.1)'}
        ],
        'priority': 1,
        'processing_method': 'closest_temporal'
    },
    
    {
        'name': 'WorldClim BIO Variables V1', 
        'id': 'WORLDCLIM/V1/BIO', 
        'type': 'Image', 
        'is_temporal': False,
        'availability': '1960-01-01 a 1991-01-01', 
        'pixel_size': 1000, 
        'prefix': 'wclim_bio',
        'bands': [
            {'band_name': 'bio01', 'unit': '°C', 'description': 'Annual Mean Temperature (Scale 0.1)'},
            {'band_name': 'bio02', 'unit': '°C', 'description': 'Mean Diurnal Range (Scale 0.1)'},
            {'band_name': 'bio03', 'unit': '%', 'description': 'Isothermality (bio02/bio07) * 100'},
            {'band_name': 'bio04', 'unit': '°C', 'description': 'Temperature Seasonality (std dev * 100)'},
            {'band_name': 'bio05', 'unit': '°C', 'description': 'Max Temperature of Warmest Month (Scale 0.1)'},
            {'band_name': 'bio06', 'unit': '°C', 'description': 'Min Temperature of Coldest Month (Scale 0.1)'},
            {'band_name': 'bio07', 'unit': '°C', 'description': 'Temperature Annual Range (bio05-bio06) (Scale 0.1)'},
            {'band_name': 'bio08', 'unit': '°C', 'description': 'Mean Temperature of Wettest Quarter (Scale 0.1)'},
            {'band_name': 'bio09', 'unit': '°C', 'description': 'Mean Temperature of Driest Quarter (Scale 0.1)'},
            {'band_name': 'bio10', 'unit': '°C', 'description': 'Mean Temperature of Warmest Quarter (Scale 0.1)'},
            {'band_name': 'bio11', 'unit': '°C', 'description': 'Mean Temperature of Coldest Quarter (Scale 0.1)'},
            {'band_name': 'bio12', 'unit': 'mm', 'description': 'Annual Precipitation'},
            {'band_name': 'bio13', 'unit': 'mm', 'description': 'Precipitation of Wettest Month'},
            {'band_name': 'bio14', 'unit': 'mm', 'description': 'Precipitation of Driest Month'},
            {'band_name': 'bio15', 'unit': 'C of V', 'description': 'Precipitation Seasonality (Coefficient of Variation)'},
            {'band_name': 'bio16', 'unit': 'mm', 'description': 'Precipitation of Wettest Quarter'},
            {'band_name': 'bio17', 'unit': 'mm', 'description': 'Precipitation of Driest Quarter'},
            {'band_name': 'bio18', 'unit': 'mm', 'description': 'Precipitation of Warmest Quarter'},
            {'band_name': 'bio19', 'unit': 'mm', 'description': 'Precipitation of Coldest Quarter'}
        ],
        'priority': 1,
        'processing_method': 'static'
    },

    # --- GRUPO 4: DATASETS TEMÁTICOS (EVAPOTRANSPIRACIÓN) ---
    {
        'name': 'Terra Net Evapotranspiration 8-Day Global 500m', 
        'id': 'MODIS/061/MOD16A2', 
        'type': 'ImageCollection', 
        'is_temporal': True,
        'availability': '2001-01-01 a presente', 
        'pixel_size': 500, 
        'prefix': 'mod16_et',
        'bands': [
            {'band_name': 'ET', 'unit': 'kg/m^2/8day', 'description': 'Total evapotranspiration (Scale 0.1)'},
            {'band_name': 'PET', 'unit': 'kg/m^2/8day', 'description': 'Total potential evapotranspiration (Scale 0.1)'},
            {'band_name': 'ET_QC', 'unit': 'code', 'description': 'Quality control flags for ET'},
            {'band_name': 'PLE', 'unit': 'kg/m^2/8day', 'description': 'Potential latent heat flux (Scale 0.1)'}
        ],
        'priority': 2,
        'processing_method': 'closest_temporal'
    },
]

# Ordenar datasets por prioridad
gee_datasets_metadata.sort(key=lambda x: x.get('priority', 3))

logger.info(f"✓ Configurados {len(gee_datasets_metadata)} datasets de GEE")

# ==============================================================================
# 5. FUNCIONES AUXILIARES OPTIMIZADAS
# ==============================================================================

def validate_bands_exist(image, expected_bands):
    """Valida que las bandas existan en la imagen antes de procesarla."""
    try:
        available_bands = image.bandNames().getInfo()
        valid_bands = [band for band in expected_bands if band in available_bands]
        
        if not valid_bands:
            logger.debug(f"Ninguna banda esperada encontrada. Disponibles: {available_bands}")
            return None, []
        
        if len(valid_bands) != len(expected_bands):
            missing_bands = set(expected_bands) - set(valid_bands)
            logger.debug(f"Bandas faltantes: {missing_bands}")
        
        return image.select(valid_bands), valid_bands
        
    except Exception as e:
        logger.error(f"Error validando bandas: {e}")
        return None, []

def get_closest_image_robust(collection, target_date_ee, max_days_window=365):
    """Versión robusta para obtener imagen más cercana a fecha objetivo."""
    try:
        collection_size = collection.size().getInfo()
        if collection_size == 0:
            logger.debug("Colección vacía")
            return None

        if target_date_ee is None:
            return collection.sort('system:time_start', False).first()

        # Estrategia de ventana expandible
        for window_days in [30, 90, 180, 365]:
            start_date = target_date_ee.advance(-window_days, 'day')
            end_date = target_date_ee.advance(window_days, 'day')
            
            filtered_collection = collection.filterDate(start_date, end_date)
            filtered_size = filtered_collection.size().getInfo()
            
            if filtered_size > 0:
                logger.debug(f"Encontradas {filtered_size} imágenes en ventana de {window_days} días")
                
                def set_time_diff(image):
                    return image.set('time_diff', 
                        ee.Number(image.get('system:time_start')).subtract(target_date_ee.millis()).abs())
                
                return filtered_collection.map(set_time_diff).sort('time_diff').first()
        
        logger.debug("No se encontraron imágenes en ventanas temporales, usando la más reciente")
        return collection.sort('system:time_start', False).first()

    except Exception as e:
        logger.error(f"Error en get_closest_image_robust: {e}")
        try:
            return collection.sort('system:time_start', False).first()
        except:
            return None

def safe_get_image_date_robust(image):
    """Obtiene fecha de imagen de manera robusta."""
    try:
        if image is None:
            return 'N/A'
        
        try:
            system_time = image.get('system:time_start')
            timestamp = system_time.getInfo()
            
            if timestamp and timestamp > 0:
                return datetime.fromtimestamp(timestamp / 1000, tz=UTC).strftime('%Y-%m-%d')
        except:
            pass
        
        try:
            system_time = image.get('system:time_end')
            timestamp = system_time.getInfo()
            
            if timestamp and timestamp > 0:
                return datetime.fromtimestamp(timestamp / 1000, tz=UTC).strftime('%Y-%m-%d')
        except:
            pass
        
        return 'N/A (Static)'
        
    except Exception as e:
        logger.debug(f"Error obteniendo fecha de imagen: {e}")
        return 'N/A (Static)'

def process_asset_for_point_optimized(asset, point_geom, target_date_ee, point_id="Unknown"):
    """Versión optimizada del procesamiento de assets."""
    asset_prefix = asset['prefix']
    result_data = {}
    
    try:
        logger.debug(f"Procesando asset '{asset['name']}' para punto {point_id}")
        
        image_to_sample = None
        
        if asset['type'] == 'ImageCollection':
            collection = ee.ImageCollection(asset['id'])
            collection = collection.filterBounds(point_geom)
            
            processing_method = asset.get('processing_method', 'default')
            
            if processing_method == 'mosaic':
                image_to_sample = collection.mosaic()
            elif processing_method == 'closest_temporal' and asset['is_temporal']:
                image_to_sample = get_closest_image_robust(collection, target_date_ee)
            else:
                if asset['is_temporal'] and target_date_ee:
                    image_to_sample = get_closest_image_robust(collection, target_date_ee)
                else:
                    image_to_sample = collection.mosaic()
                    
        elif asset['type'] == 'Image':
            image_to_sample = ee.Image(asset['id'])

        if image_to_sample is None:
            raise Exception("No se pudo obtener imagen del asset")

        # Validar y seleccionar bandas
        expected_bands = [band['band_name'] for band in asset['bands']]
        validated_image, valid_bands = validate_bands_exist(image_to_sample, expected_bands)
        
        if validated_image is None:
            raise Exception("No se encontraron bandas válidas")

        # Extraer datos con parámetros optimizados
        try:
            data = validated_image.reduceRegion(
                reducer=ee.Reducer.first(),
                geometry=point_geom,
                scale=asset['pixel_size'],
                maxPixels=1e9,
                tileScale=4,  # Usar tileScale más alto para mejor rendimiento
                bestEffort=True
            ).getInfo()
            
        except Exception as reduce_error:
            logger.debug(f"Error en reduceRegion, intentando parámetros alternativos: {reduce_error}")
            
            data = validated_image.reduceRegion(
                reducer=ee.Reducer.first(),
                geometry=point_geom,
                scale=asset['pixel_size'] * 2,
                maxPixels=1e8,
                tileScale=2,
                bestEffort=True
            ).getInfo()

        # Obtener fecha de la imagen
        rs_date = safe_get_image_date_robust(image_to_sample)
        
        # Llenar datos de resultado
        for band_info in asset['bands']:
            b_name = band_info['band_name']
            
            if b_name in valid_bands:
                value = data.get(b_name)
                if value is not None and not (isinstance(value, float) and pd.isna(value)):
                    result_data[f'{asset_prefix}_{b_name}_value'] = value
                else:
                    result_data[f'{asset_prefix}_{b_name}_value'] = None
            else:
                result_data[f'{asset_prefix}_{b_name}_value'] = None
            
            # Metadatos de banda
            result_data[f'{asset_prefix}_{b_name}_unit'] = band_info['unit']
            result_data[f'{asset_prefix}_{b_name}_description'] = band_info['description']
        
        # Metadatos generales del asset
        result_data[f'{asset_prefix}_rs_date'] = rs_date
        result_data[f'{asset_prefix}_asset_id'] = asset['id']
        result_data[f'{asset_prefix}_pixel_size'] = asset['pixel_size']
        result_data[f'{asset_prefix}_processing_method'] = asset.get('processing_method', 'default')
        result_data[f'{asset_prefix}_status'] = 'success'
        
        logger.debug(f"✓ Asset '{asset['name']}' procesado exitosamente para punto {point_id}")
        return result_data
        
    except Exception as e:
        error_msg = f"Error procesando asset '{asset['name']}': {str(e)}"
        logger.error(f"✗ {error_msg} para punto {point_id}")
        
        # Llenar con valores nulos en caso de error
        for band_info in asset['bands']:
            b_name = band_info['band_name']
            result_data[f'{asset_prefix}_{b_name}_value'] = None
            result_data[f'{asset_prefix}_{b_name}_unit'] = band_info['unit']
            result_data[f'{asset_prefix}_{b_name}_description'] = band_info['description']
        
        result_data[f'{asset_prefix}_rs_date'] = 'N/A'
        result_data[f'{asset_prefix}_asset_id'] = asset['id']
        result_data[f'{asset_prefix}_pixel_size'] = asset['pixel_size']
        result_data[f'{asset_prefix}_processing_method'] = asset.get('processing_method', 'default')
        result_data[f'{asset_prefix}_status'] = f'error: {str(e)[:100]}'
        
        return result_data

def process_single_point_with_retry(point_data, max_retries=MAX_RETRIES):
    """Procesa un punto individual con reintentos automáticos."""
    point_id = point_data['ID']
    
    for attempt in range(max_retries):
        try:
            result = process_single_point_optimized(point_data)
            if result is not None:
                return result
            else:
                logger.warning(f"Intento {attempt + 1}/{max_retries} falló para punto {point_id} (resultado None)")
                
        except Exception as e:
            logger.warning(f"Intento {attempt + 1}/{max_retries} falló para punto {point_id}: {str(e)}")
            
        if attempt < max_retries - 1:
            wait_time = 2 ** attempt  # Backoff exponencial
            time.sleep(wait_time)
    
    # Si todos los intentos fallan, crear resultado de error
    logger.error(f"✗ Todos los intentos fallaron para punto {point_id}")
    error_result = {'ID': point_id}
    
    for asset in gee_datasets_metadata:
        for band_info in asset['bands']:
            b_name = band_info['band_name']
            asset_prefix = asset['prefix']
            error_result[f'{asset_prefix}_{b_name}_value'] = None
            error_result[f'{asset_prefix}_{b_name}_unit'] = band_info['unit']
            error_result[f'{asset_prefix}_{b_name}_description'] = band_info['description']
        
        error_result[f'{asset_prefix}_rs_date'] = 'N/A'
        error_result[f'{asset_prefix}_asset_id'] = asset['id']
        error_result[f'{asset_prefix}_pixel_size'] = asset['pixel_size']
        error_result[f'{asset_prefix}_processing_method'] = asset.get('processing_method', 'default')
        error_result[f'{asset_prefix}_status'] = 'max_retries_exceeded'
    
    return error_result

def process_single_point_optimized(point_data):
    """Procesa un punto individual de manera optimizada."""
    try:
        point_id = point_data['ID']
        lat = point_data['Latitude_GCS_WGS_1984']
        lon = point_data['Longitude_GCS_WGS_1984']
        
        # Validar coordenadas
        if pd.isna(lat) or pd.isna(lon):
            raise ValueError(f"Coordenadas inválidas para punto {point_id}: lat={lat}, lon={lon}")
        
        point_geom = ee.Geometry.Point([lon, lat])
        
        # Procesar fecha
        target_date_ee = None
        if 'Date_String_dt' in point_data and pd.notna(point_data['Date_String_dt']):
            try:
                date_obj = point_data['Date_String_dt']
                target_date_ee = ee.Date(date_obj.strftime('%Y-%m-%d'))
            except Exception as date_error:
                logger.debug(f"Error procesando fecha para punto {point_id}: {date_error}")
        
        # Inicializar resultado con datos básicos
        result = {
            'ID': point_id,
            'original_latitude': lat,
            'original_longitude': lon,
            'original_date': point_data.get('Date_String', 'N/A'),
            'processing_timestamp': datetime.now().isoformat(),
        }
        
        # Procesar cada asset
        for asset in gee_datasets_metadata:
            try:
                asset_result = process_asset_for_point_optimized(
                    asset, point_geom, target_date_ee, point_id
                )
                result.update(asset_result)
                
            except Exception as asset_error:
                logger.error(f"Error procesando asset '{asset['name']}' para punto {point_id}: {asset_error}")
                
                # Llenar con valores de error para este asset
                asset_prefix = asset['prefix']
                for band_info in asset['bands']:
                    b_name = band_info['band_name']
                    result[f'{asset_prefix}_{b_name}_value'] = None
                    result[f'{asset_prefix}_{b_name}_unit'] = band_info['unit']
                    result[f'{asset_prefix}_{b_name}_description'] = band_info['description']
                
                result[f'{asset_prefix}_rs_date'] = 'N/A'
                result[f'{asset_prefix}_asset_id'] = asset['id']
                result[f'{asset_prefix}_pixel_size'] = asset['pixel_size']
                result[f'{asset_prefix}_processing_method'] = asset.get('processing_method', 'default')
                result[f'{asset_prefix}_status'] = f'asset_error: {str(asset_error)[:100]}'
        
        return result
        
    except Exception as e:
        logger.error(f"Error crítico procesando punto {point_data.get('ID', 'Unknown')}: {e}")
        return None

# ==============================================================================
# 6. FUNCIONES DE PROCESAMIENTO PARALELO
# ==============================================================================

def save_results_safely(results_list, output_path):
    """Guarda resultados de manera segura con manejo de errores."""
    try:
        with save_lock:
            if not results_list:
                logger.warning("Lista de resultados vacía, no se guardará archivo")
                return False
                
            df_results = pd.DataFrame(results_list)
            
            # Información de guardado
            logger.info(f"Guardando {len(df_results)} resultados en: {output_path}")
            
            # Guardar con configuración robusta
            with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
                df_results.to_excel(writer, sheet_name='GEE_Results', index=False)
            
            logger.info(f"✓ Archivo guardado exitosamente: {output_path}")
            return True
            
    except Exception as e:
        logger.error(f"✗ Error guardando resultados: {e}")
        logger.error(traceback.format_exc())
        return False

def process_batch_parallel(batch_data):
    """Procesa un lote de puntos en paralelo."""
    batch_results = []
    batch_size = len(batch_data)
    
    logger.info(f"Procesando lote de {batch_size} puntos...")
    
    # Usar ThreadPoolExecutor para procesamiento paralelo
    with concurrent.futures.ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        # Enviar todas las tareas
        future_to_point = {
            executor.submit(process_single_point_with_retry, point_data): point_data['ID'] 
            for _, point_data in batch_data.iterrows()
        }
        
        # Recoger resultados conforme se completan
        for future in concurrent.futures.as_completed(future_to_point, timeout=TIMEOUT_SECONDS * batch_size):
            point_id = future_to_point[future]
            
            try:
                result = future.result(timeout=TIMEOUT_SECONDS)
                if result:
                    batch_results.append(result)
                    logger.debug(f"✓ Punto {point_id} procesado exitosamente")
                else:
                    logger.warning(f"✗ Punto {point_id} devolvió resultado None")
                    
            except concurrent.futures.TimeoutError:
                logger.error(f"✗ Timeout procesando punto {point_id}")
                progress_tracker.add_error(f"Timeout en punto {point_id}")
                
            except Exception as e:
                logger.error(f"✗ Error procesando punto {point_id}: {e}")
                progress_tracker.add_error(f"Error en punto {point_id}: {str(e)}")
    
    logger.info(f"Lote completado: {len(batch_results)}/{batch_size} puntos exitosos")
    return batch_results

# ==============================================================================
# 7. FUNCIÓN PRINCIPAL DE PROCESAMIENTO
# ==============================================================================

def main_processing_loop():
    """Bucle principal de procesamiento con manejo robusto de errores."""
    global final_results_list
    
    logger.info("="*80)
    logger.info("INICIANDO PROCESAMIENTO PRINCIPAL")
    logger.info("="*80)
    
    total_batches = (total_points_to_process + BATCH_SIZE - 1) // BATCH_SIZE
    current_batch_num = 0
    
    try:
        # Procesar en lotes
        for batch_start in range(0, total_points_to_process, BATCH_SIZE):
            current_batch_num += 1
            batch_end = min(batch_start + BATCH_SIZE, total_points_to_process)
            batch_data = df_points_to_process.iloc[batch_start:batch_end].copy()
            
            logger.info(f"\n--- LOTE {current_batch_num}/{total_batches} ---")
            logger.info(f"Procesando puntos {batch_start + start_index} a {batch_end + start_index - 1}")
            
            batch_start_time = time.time()
            
            try:
                # Procesar lote
                batch_results = process_batch_parallel(batch_data)
                
                # Agregar resultados a la lista principal
                final_results_list.extend(batch_results)
                
                # Actualizar progreso
                total_processed = len(final_results_list)
                successful_in_batch = len(batch_results)
                batch_time = time.time() - batch_start_time
                
                progress_tracker.update(
                    processed_points=total_processed,
                    successful_points=total_processed,  # Asumiendo que todos los agregados son exitosos
                    current_batch=current_batch_num,
                    last_processed_id=final_results_list[-1]['ID'] if final_results_list else None
                )
                
                logger.info(f"✓ Lote {current_batch_num} completado en {batch_time:.2f}s")
                logger.info(f"  - Exitosos: {successful_in_batch}/{len(batch_data)}")
                logger.info(f"  - Total procesados: {total_processed}")
                logger.info(f"  - Progreso: {(total_processed/len(df_points)*100):.1f}%")
                
                # Guardar resultados periódicamente
                if current_batch_num % SAVE_INTERVAL == 0 or current_batch_num == total_batches:
                    logger.info(f"PROGRESO: Guardando resultados (Lote {current_batch_num})")
                    
                    # Guardar archivo principal
                    save_success = save_results_safely(final_results_list, output_excel_path)
                    
                    # Guardar checkpoint
                    checkpoint_success = save_checkpoint(
                        final_results_list, 
                        batch_end + start_index - 1, 
                        checkpoint_file
                    )
                    
                    if save_success and checkpoint_success:
                        logger.info(f"✓ Guardado y checkpoint exitosos")
                    else:
                        logger.warning(f"⚠ Problemas en guardado o checkpoint")
                
            except Exception as batch_error:
                logger.error(f"✗ Error crítico en lote {current_batch_num}: {batch_error}")
                logger.error(traceback.format_exc())
                progress_tracker.add_error(f"Error en lote {current_batch_num}: {str(batch_error)}")
                
                # Continuar con el siguiente lote
                continue
        
        # Procesamiento completado
        logger.info("\n" + "="*80)
        logger.info("PROCESAMIENTO COMPLETADO")
        logger.info("="*80)
        
        total_processed = len(final_results_list)
        success_rate = (total_processed / len(df_points)) * 100 if len(df_points) > 0 else 0
        
        logger.info(f"📊 ESTADÍSTICAS FINALES:")
        logger.info(f"  - Total de puntos en dataset: {len(df_points)}")
        logger.info(f"  - Puntos procesados exitosamente: {total_processed}")
        logger.info(f"  - Tasa de éxito: {success_rate:.1f}%")
        logger.info(f"  - Archivo de salida: {output_excel_path}")
        
        # Guardar resultados finales
        final_save_success = save_results_safely(final_results_list, output_excel_path)
        
        if final_save_success:
            logger.info(f"✅ ÉXITO: Procesamiento completado y resultados guardados")
        else:
            logger.error(f"❌ ERROR: Procesamiento completado pero falló el guardado final")
        
        # Actualizar progreso final
        progress_tracker.update(
            processed_points=total_processed,
            successful_points=total_processed,
            current_batch=total_batches
        )
        
        return True
        
    except Exception as critical_error:
        logger.error(f"💥 ERROR CRÍTICO EN PROCESAMIENTO PRINCIPAL: {critical_error}")
        logger.error(traceback.format_exc())
        progress_tracker.add_error(f"Error crítico: {str(critical_error)}")
        
        # Intentar guardar resultados parciales
        if final_results_list:
            logger.info("🆘 Intentando guardar resultados parciales...")
            partial_output_path = output_folder / f"resultados_parciales_{timestamp}.xlsx"
            save_results_safely(final_results_list, partial_output_path)
        
        return False

# ==============================================================================
# 8. PUNTO DE ENTRADA PRINCIPAL
# ==============================================================================

if __name__ == "__main__":
    try:
        logger.info("🚀 INICIANDO EXTRACCIÓN DE DATOS DE GOOGLE EARTH ENGINE")
        logger.info(f"📁 Directorio base: {BASE_DIR}")
        logger.info(f"📊 Archivo de entrada: {input_excel_path}")
        logger.info(f"📈 Archivo de salida: {output_excel_path}")
        logger.info(f"🔍 Total de datasets GEE: {len(gee_datasets_metadata)}")
        
        # Mostrar configuración de datasets
        logger.info("\n📋 DATASETS CONFIGURADOS:")
        for i, asset in enumerate(gee_datasets_metadata, 1):
            logger.info(f"  {i}. {asset['name']} ({asset['pixel_size']}m)")
        
        # Iniciar procesamiento principal
        main_start_time = time.time()
        success = main_processing_loop()
        total_time = time.time() - main_start_time
        
        # Resumen final
        logger.info("\n" + "="*80)
        if success:
            logger.info("🎉 PROCESO COMPLETADO EXITOSAMENTE")
        else:
            logger.info("⚠️  PROCESO COMPLETADO CON ERRORES")
        
        logger.info(f"⏱️  Tiempo total de ejecución: {total_time/60:.1f} minutos")
        logger.info(f"📂 Resultados guardados en: {output_excel_path}")
        logger.info(f"📋 Logs guardados en: {logs_folder}")
        logger.info("="*80)
        
    except Exception as startup_error:
        logger.error(f"💥 ERROR CRÍTICO AL INICIAR: {startup_error}")
        logger.error(traceback.format_exc())
        sys.exit(1)